# 🏥 TRIAGE GRPO Training — Colab Edition

**Meta PyTorch OpenEnv Hackathon** | 14 datasets | 9 reward verifiers | GRPO

| Item | Value |
|---|---|
| Model | Qwen2.5-7B (4-bit NF4) |
| Datasets | 14 sources (7 HF + 6 Kaggle + 1 base) |
| Method | GRPO with 9 reward verifiers |
| Hardware | Colab T4 / Kaggle P100 |

In [ ]:
!pip install -q "transformers>=4.45" "trl>=0.12" "peft>=0.13" "bitsandbytes>=0.46" "datasets>=3.0" "accelerate>=1.0" huggingface_hub kagglehub pyarrow

In [ ]:
import json,re,random,logging,time,os,gc,torch,inspect
from pathlib import Path
logging.basicConfig(level=logging.INFO,format='%(asctime)s %(levelname)s %(message)s')
CFG={'model':'Qwen/Qwen2.5-7B','max_seq_length':512,'lora_r':16,'lora_alpha':32,'lora_dropout':0,
     'lora_targets':['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
     'num_generations':4,'max_completion_length':200,'temperature':0.9,
     'epochs':1,'batch_size':1,'grad_accum':4,'lr':5e-5,
     'logging_steps':5,'save_steps':50,'output_dir':'./grpo_output','base_dataset':'balarajr/triage-grpo'}
HF_TOKEN=os.environ.get('HF_TOKEN','')
USE_BF16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE=torch.bfloat16 if USE_BF16 else torch.float16
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Dtype: {"bf16" if USE_BF16 else "fp16"}')

In [ ]:
# 9 Reward Verifiers
_VA=frozenset({'TRIAGE_PATIENT','ASSIGN_TREATMENT','TRANSFER_TO_ICU','TRANSFER_TO_WARD','ACTIVATE_OVERFLOW','ORDER_MEDICATION','FLAG_POLICY_VIOLATION','OVERRIDE_DECISION','UPDATE_EHR','REQUEST_STAFF','VERIFY_INSURANCE'})
_RK={'action_type','target_id','priority','reasoning'}
_EV=[r'P-\d{2,3}',r'patient\s+\d+',r'\d+%',r'\d+/\d+',r'BP\s*\d+',r'HR\s*\d+',r'ICU\s+at\s+\d+',r'beds?\s+\d+',r'age\s+\d+',r'critical|immediate|urgent|stable']
_FL=['i need more information',"i'm not sure",'let me think','i cannot determine',"i don't know",'more data needed']
_FB=[r'\bimport\s+os\b',r'\bimport\s+sys\b',r'\bexec\s*\(',r'\beval\s*\(',r'\breward\s*[:=]\s*1\.0\b']

def _xj(text):
    text=text.strip()
    try: return json.loads(text)
    except: pass
    m=re.search(r'```(?:json)?\s*(\{.*?\})\s*```',text,re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    m=re.search(r'\{[^{}]*\}',text,re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except: pass
    return None

def _sp(prompt):
    s={'alive_count':20,'deceased_count':0,'critical_count':0,'icu_occupancy':0.5,'violations_injected':0,'violations_caught':0,'survival_rate':1.0}
    m=re.search(r'ICU OCCUPANCY:\s*(\d+)%',prompt)
    if m: s['icu_occupancy']=int(m.group(1))/100.0
    m=re.search(r'CRITICAL PATIENTS\s*\((\d+)',prompt)
    if m: s['critical_count']=int(m.group(1))
    m=re.search(r'VIOLATIONS INJECTED:\s*(\d+)\s*\|\s*CAUGHT:\s*(\d+)',prompt)
    if m: s['violations_injected'],s['violations_caught']=int(m.group(1)),int(m.group(2))
    m=re.search(r'SURVIVAL RATE:\s*(\d+\.?\d*)%',prompt)
    if m: s['survival_rate']=float(m.group(1))/100.0
    t=20;s['alive_count']=int(s['survival_rate']*t);s['deceased_count']=t-s['alive_count']
    return s

def reward_format_compliance(completions,**kw):
    R=[]
    for c in completions:
        p=_xj(c)
        if not p or not _RK.issubset(p.keys()): R.append(0.0);continue
        a=str(p.get('action_type','')).upper()
        if a not in _VA: R.append(0.0);continue
        try: int(p['target_id'])
        except: R.append(0.0);continue
        try:
            pr=int(p['priority'])
            if not 1<=pr<=10: R.append(0.0);continue
        except: R.append(0.0);continue
        if len(str(p.get('reasoning','')).strip())<10: R.append(0.0);continue
        R.append(1.0)
    return R

def reward_patient_survival(completions,**kw):
    P=kw.get('prompts',kw.get('prompt',['']));R=[]
    for i,c in enumerate(completions):
        s=_sp(P[i] if i<len(P) else '');t=s['alive_count']+s['deceased_count']
        R.append(s['alive_count']/t if t>0 else 1.0)
    return R

def reward_icu_efficiency(completions,**kw):
    P=kw.get('prompts',kw.get('prompt',['']));R=[]
    for i,c in enumerate(completions):
        o=_sp(P[i] if i<len(P) else '')['icu_occupancy']
        R.append(1.0 if o<=0.85 else (1.0-(o-0.85)*5.0 if o<=0.95 else max(0.0,0.5-(o-0.95)*10.0)))
    return R

def reward_violation_detection(completions,**kw):
    P=kw.get('prompts',kw.get('prompt',['']));R=[]
    for i,c in enumerate(completions):
        s=_sp(P[i] if i<len(P) else '');inj,ct=s['violations_injected'],s['violations_caught']
        R.append(min(1.0,ct/max(inj,1)) if inj>0 else 1.0)
    return R

def reward_reasoning_quality(completions,**kw):
    R=[]
    for c in completions:
        p=_xj(c)
        if not p: R.append(0.0);continue
        r=str(p.get('reasoning',''))
        if len(r)<20: R.append(0.1);continue
        ev=sum(1 for pat in _EV if re.search(pat,r,re.I))
        if any(f in r.lower() for f in _FL): R.append(0.1);continue
        R.append(min(1.0,0.3+min(0.7,ev*0.15)))
    return R

def reward_response_speed(completions,**kw):
    return [1.0 if len(c)<=400 else (1.0-(len(c)-400)*0.001 if len(c)<=800 else max(0.2,0.6-(len(c)-800)*0.0005)) for c in completions]

def reward_no_hallucination(completions,**kw):
    P=kw.get('prompts',kw.get('prompt',['']));R=[]
    for i,c in enumerate(completions):
        p=_xj(c)
        if not p: R.append(0.5);continue
        mn={int(m.group(1)) for m in re.finditer(r'P-(\d{2,3})',str(p.get('reasoning','')),re.I)}
        if not mn: R.append(1.0);continue
        vl={int(m.group(1)) for m in re.finditer(r'P-(\d{2,3})',P[i] if i<len(P) else '')}
        R.append(0.0 if mn-vl else 1.0)
    return R

def reward_action_alignment(completions,**kw):
    P=kw.get('prompts',kw.get('prompt',['']));R=[]
    for i,c in enumerate(completions):
        p=_xj(c)
        if not p: R.append(0.0);continue
        s=_sp(P[i] if i<len(P) else '');a=str(p.get('action_type','')).upper()
        o,cr=s['icu_occupancy'],s['critical_count'];v=s['violations_injected']-s['violations_caught']
        sm={'TRIAGE_PATIENT':1.0 if cr>0 else 0.5,'TRANSFER_TO_ICU':1.0 if o<0.9 and cr>0 else 0.3,'ACTIVATE_OVERFLOW':1.0 if o>=0.85 else 0.2,'FLAG_POLICY_VIOLATION':1.0 if v>0 else 0.4,'ORDER_MEDICATION':0.8 if cr>0 else 0.5,'ASSIGN_TREATMENT':0.9 if cr>0 else 0.5}
        R.append(sm.get(a,0.5))
    return R

def reward_sandbox_safety(completions,**kw):
    R=[]
    for c in completions:
        safe=all(not re.search(pat,c,re.I) for pat in _FB) and len(c)<=3000
        w=c.split()
        if len(w)>20 and len(set(w))/len(w)<0.2: safe=False
        R.append(1.0 if safe else 0.0)
    return R

REWARD_FUNCS=[reward_format_compliance,reward_patient_survival,reward_icu_efficiency,reward_violation_detection,reward_reasoning_quality,reward_response_speed,reward_no_hallucination,reward_action_alignment,reward_sandbox_safety]
print(f'Loaded {len(REWARD_FUNCS)} reward verifiers')

In [ ]:
# Cell 4: Load Model + LoRA
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=COMPUTE_DTYPE,bnb_4bit_use_double_quant=True)
print(f'Loading {CFG["model"]} ...')
model=AutoModelForCausalLM.from_pretrained(CFG['model'],quantization_config=bnb,device_map='auto',torch_dtype=COMPUTE_DTYPE,trust_remote_code=True,token=HF_TOKEN or None)
tokenizer=AutoTokenizer.from_pretrained(CFG['model'],trust_remote_code=True,token=HF_TOKEN or None)
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token;model.config.pad_token_id=model.config.eos_token_id
model=prepare_model_for_kbit_training(model)
lora=LoraConfig(r=CFG['lora_r'],lora_alpha=CFG['lora_alpha'],lora_dropout=CFG['lora_dropout'],target_modules=CFG['lora_targets'],bias='none',task_type='CAUSAL_LM')
model=get_peft_model(model,lora)
model.print_trainable_parameters()

## Cell 5: Multi-Source Dataset Pipeline (14 Sources)
7 HuggingFace + 6 Kaggle + 1 base triage dataset

In [ ]:
import pandas as pd,kagglehub
from datasets import load_dataset,Dataset
rng=random.Random(42)
AG=['ER_TRIAGE','ICU_MANAGEMENT','PHARMACY','CMO_OVERSIGHT','HR_ROSTERING','IT_SYSTEMS']
CR=['MASS_CASUALTY','OUTBREAK','EQUIPMENT_FAILURE','STAFF_SHORTAGE']
JS='Respond with ONLY valid JSON:\n{\n  "action_type": "<action>",\n  "target_id": <int>,\n  "priority": <1-10>,\n  "reasoning": "<cite data>"\n}'

def mh():
    a=rng.choice(AG);c=rng.choice(CR);icu=rng.randint(30,98);cr=rng.randint(1,10)
    return (f'You are the {a} agent in a hospital crisis simulation.\n\nCRISIS: {c}\nSTEP: {rng.randint(0,19)}/20\n'
            f'ICU OCCUPANCY: {icu}% ({icu*20//100}/20 beds)\nCRITICAL PATIENTS ({cr} total):\n'
            f'  P-{rng.randint(1,99):03d}: CRITICAL -- BP {rng.randint(55,95)}/{rng.randint(25,65)}, HR {rng.randint(95,160)}\n'
            f'VIOLATIONS INJECTED: {rng.randint(0,5)} | CAUGHT: {rng.randint(0,3)}\nSURVIVAL RATE: {rng.uniform(82,100):.1f}%\n\n')

all_prompts=[]

# [0] Base triage dataset
print('[0/14] balarajr/triage-grpo')
try:
    ds=load_dataset(CFG['base_dataset'],split='train',token=HF_TOKEN or None);all_prompts.extend(list(ds['prompt']))
    print(f'  \u2714 {len(ds)}')
except Exception as e: print(f'  \u2718 {e}')

# HuggingFace datasets
HF_SOURCES=[
    ('FreedomIntelligence/medical-o1-reasoning-SFT','data/train-00000-of-00001.parquet','CLINICAL',500),
    ('bigbio/med_qa','med_qa_en_bigbio_qa/train-00000-of-00001.parquet','MEDQA',500),
    ('sdiazlor/medical-reasoning-dataset','data/train-00000-of-00001.parquet','REASONING',500),
    ('Anthropic/hh-rlhf','data/harmless-base/train-00000-of-00001.parquet','SAFETY',300),
    ('PKU-Alignment/PKU-SafeRLHF','data/train-00000-of-00001.parquet','ALIGNMENT',300),
    ('lavita/ChatDoctor-iCliniq','data/train-00000-of-00001.parquet','CONSULT',500),
]
for i,(slug,path,tag,n) in enumerate(HF_SOURCES,1):
    print(f'[{i}/14] {slug}')
    try:
        df=pd.read_parquet(f'hf://datasets/{slug}/{path}')
        for _,row in df.sample(min(n,len(df)),random_state=42).iterrows():
            txt=str(row.get('question',row.get('input',row.get('instruction',row.get('chosen',row.get('prompt',''))))))[:300]
            all_prompts.append(mh()+f'{tag}: {txt}\n\n{JS}')
        print(f'  \u2714 {min(n,len(df))}/{len(df)}')
    except Exception as e: print(f'  \u2718 {e}')

# [7] medical flashcards (JSON format)
print('[7/14] medalpaca/medical_meadow_medical_flashcards')
try:
    df=pd.read_json('hf://datasets/medalpaca/medical_meadow_medical_flashcards/medical_meadow_medical_flashcards.json')
    for _,row in df.sample(min(500,len(df)),random_state=42).iterrows():
        txt=str(row.get('input',row.get('instruction','')))[:300]
        all_prompts.append(mh()+f'FLASHCARD: {txt}\n\n{JS}')
    print(f'  \u2714 500/{len(df)}')
except Exception as e: print(f'  \u2718 {e}')

# Kaggle datasets
KG_SOURCES=[
    ('thedevastator/medical-q-a-structured','STRUCT_QA',500),
    ('nehaprabhavalkar/av-healthcare-analytics-ii','ANALYTICS',500),
    ('jpmiller/layoutlm','NLP_REC',300),
    ('thedevastator/usmle-medical-licensing-examination','USMLE',500),
    ('kaushil268/disease-prediction-using-machine-learning','DISEASE',500),
    ('maalona/hospital-triage-and-patient-history-data','TRIAGE_HIST',500),
]
for i,(slug,tag,n) in enumerate(KG_SOURCES,8):
    print(f'[{i}/14] Kaggle: {slug}')
    try:
        p=kagglehub.dataset_download(slug)
        csvs=[f for f in os.listdir(p) if f.endswith('.csv')]
        if not csvs: print('  \u2718 no CSV');continue
        df=pd.read_csv(os.path.join(p,csvs[0]),nrows=n*3)
        for _,row in df.sample(min(n,len(df)),random_state=42).iterrows():
            cols=' | '.join(f'{c}: {row[c]}' for c in df.columns[:6])[:300]
            all_prompts.append(mh()+f'{tag}: {cols}\n\n{JS}')
        print(f'  \u2714 {min(n,len(df))}/{len(df)}')
    except Exception as e: print(f'  \u2718 {e}')

random.shuffle(all_prompts)
dataset=Dataset.from_dict({'prompt':all_prompts})
print(f'\n\u2501'*60)
print(f'\u2705 TOTAL: {len(dataset)} prompts from 14 sources')

In [ ]:
# Cell 6: GRPO Training
from trl import GRPOTrainer,GRPOConfig
gk=dict(output_dir=CFG['output_dir'],num_train_epochs=CFG['epochs'],per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],learning_rate=CFG['lr'],max_completion_length=CFG['max_completion_length'],
    num_generations=CFG['num_generations'],temperature=CFG['temperature'],logging_steps=CFG['logging_steps'],
    save_steps=CFG['save_steps'],save_total_limit=2,report_to='none',bf16=USE_BF16,fp16=not USE_BF16,seed=42)
sig=inspect.signature(GRPOConfig)
if not any(p.kind==inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()):
    gk={k:v for k,v in gk.items() if k in set(sig.parameters)}
args=GRPOConfig(**gk)
tk=dict(model=model,reward_funcs=REWARD_FUNCS,args=args,train_dataset=dataset)
tp=set(inspect.signature(GRPOTrainer.__init__).parameters)
if 'processing_class' in tp: tk['processing_class']=tokenizer
elif 'tokenizer' in tp: tk['tokenizer']=tokenizer
trainer=GRPOTrainer(**tk)
print(f'Starting GRPO ({len(dataset)} prompts, {CFG["epochs"]} epochs)...')
result=trainer.train()
print(f'Done! Loss={result.training_loss:.4f} Steps={result.global_step}')
trainer.save_model(CFG['output_dir']);tokenizer.save_pretrained(CFG['output_dir'])

In [ ]:
# Cell 7: Quick Evaluation
from peft import PeftModel
del model;gc.collect();torch.cuda.empty_cache()
base=AutoModelForCausalLM.from_pretrained(CFG['model'],quantization_config=bnb,device_map='auto',torch_dtype=COMPUTE_DTYPE,trust_remote_code=True,token=HF_TOKEN or None)
model=PeftModel.from_pretrained(base,CFG['output_dir'])
tok=AutoTokenizer.from_pretrained(CFG['output_dir'],trust_remote_code=True)
tp=('You are the ER_TRIAGE agent in a hospital crisis simulation.\n\nCRISIS: MASS_CASUALTY\nSTEP: 5/20\n'
    'ICU OCCUPANCY: 85% (17/20 beds)\nCRITICAL PATIENTS (3 total):\n  P-042: CRITICAL -- BP 72/40, HR 140\n'
    '  P-019: CRITICAL -- BP 65/35, HR 155\n  P-067: CRITICAL -- BP 80/50, HR 120\n'
    'VIOLATIONS INJECTED: 2 | CAUGHT: 1\nSURVIVAL RATE: 90.0%\n\n'
    'Respond with ONLY valid JSON:\n{\n  "action_type": "<action>",\n  "target_id": <int>,\n  "priority": <1-10>,\n  "reasoning": "<cite data>"\n}')
inputs=tok(tp,return_tensors='pt').to(model.device)
with torch.no_grad():
    out=model.generate(**inputs,max_new_tokens=200,temperature=0.7,do_sample=True)
resp=tok.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True)
print('='*60);print('EVAL RESPONSE:');print(resp);print('='*60)
parsed=_xj(resp)
if parsed:
    scores={fn.__name__:fn([resp],prompts=[tp])[0] for fn in REWARD_FUNCS}
    total=sum(scores.values())/len(scores)*100
    print(f'\nReward Scores: {json.dumps(scores,indent=2)}')
    print(f'Overall: {total:.0f}/100')
else: print('WARNING: Could not parse JSON')